In [1]:
import sys
import pandas as pd
sys.path.append('../Source')

from Pipeline.clean_ACLED import cleanACLED
from Pipeline.clean_market import cleanMarket

acledData = cleanACLED('../Data/raw/ACLED Data_2026-06-24.csv')
marketData = cleanMarket('../data/raw/rawMarket.csv')

print(f'ACLED: {acledData.shape[0]:,} rows')
print(f'Market: {marketData.shape[0]:,} trading days')
print(f'Market date range: {marketData.index.min()} to {marketData.index.max()}')


ACLED: 1,087,171 rows
Market: 2,129 trading days
Market date range: 2018-01-02 00:00:00 to 2026-06-23 00:00:00


In [2]:
tradingDays = pd.DatetimeIndex(marketData.index)

print(f'Total trading days: {len(tradingDays):,}')
print(f'First trading day: {tradingDays[0]}')
print(f'Last trading day: {tradingDays[-1]}')

weekends = tradingDays[tradingDays.weekday >= 5]
print(f'Weekend days in trading calendar: {len(weekends)}')

Total trading days: 2,129
First trading day: 2018-01-02 00:00:00
Last trading day: 2026-06-23 00:00:00
Weekend days in trading calendar: 0


In [3]:
def matchToTradingDay(eventDate, tradingDays): 
    """
    Match an event date to the nearest trading day.
    
    Arguments:
        eventDate: The date of the event.
    tradingDays: The index of valid trading days.
    
    Returns:
        The nearest trading day 
    
    """
    index = tradingDays.searchsorted(eventDate)
    index = min(index, len(tradingDays) - 1)
    return tradingDays[index]

testDates = ['2022-02-26', '2022-02-27', '2022-02-28']
for date in testDates: 
    matchedDate = matchToTradingDay(pd.Timestamp(date), tradingDays)
    print(f'{date} {pd.Timestamp(date).day_name()} -> {matchedDate.date()}')

2022-02-26 Saturday -> 2022-02-28
2022-02-27 Sunday -> 2022-02-28
2022-02-28 Monday -> 2022-02-28


In [4]:
acledData['tradingDay'] = acledData['event_date'].apply(
    lambda x: matchToTradingDay(x, tradingDays)
)

print('tradingDay column added to ACLED data')
print(f'Sample of matched days: ')
acledData[['event_date', 'tradingDay']].head(10)

tradingDay column added to ACLED data
Sample of matched days: 


,event_date,tradingDay
0,2018-01-01,2018-01-02
1,2018-01-01,2018-01-02
2,2018-01-01,2018-01-02
3,2018-01-01,2018-01-02
4,2018-01-01,2018-01-02
5,2018-01-01,2018-01-02
6,2018-01-01,2018-01-02
7,2018-01-01,2018-01-02
8,2018-01-01,2018-01-02
9,2018-01-01,2018-01-02


In [5]:
# Compute daily event counts per region
dailyCounts = acledData.groupby(['tradingDay', 'region']).size().unstack(fill_value=0)

print(f'Shape: {dailyCounts.shape}')
print(f'Regions covered: ')
for region in dailyCounts.columns: 
    print(f'  {region}')
dailyCounts.head()

Shape: (1879, 4)
Regions covered: 
  Caucasus and Central Asia
  Europe
  Middle East
  Northern Africa


region,Caucasus and Central Asia,Europe,Middle East,Northern Africa
tradingDay,,,,
2018-01-02,158,56,376,11
2018-01-03,60,27,180,14
2018-01-04,82,32,150,15
2018-01-05,68,32,192,25
2018-01-08,232,99,542,75


In [6]:
closeColumns = ['Close_EEM', 'Close_GLD', 'Close_ITA', 'Close_USO', 'Close_VIX']
marketReturns = marketData[closeColumns].pct_change()

marketReturns.columns = [col.replace('Close_', 'Return_') for col in marketReturns.columns]

print(f'Shape: {marketReturns.shape}')
marketReturns.head()

Shape: (2129, 5)


,Return_EEM,Return_GLD,Return_ITA,Return_USO,Return_VIX
Date,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN
2018-01-03,0.009581,-0.002637,0.001486,0.022370,-0.063460
2018-01-04,0.004952,0.005127,0.006995,0.002431,0.007650
2018-01-05,0.008623,-0.001036,0.008998,-0.004850,0.000000
2018-01-08,0.000000,-0.000160,0.006050,0.005686,0.032538


In [7]:
def getEventWindow(tradingDay, marketReturns, before=1, after=3):
    """
    Extract returns for a [-before, +after] window around a given trading day.

    Arguments:
        tradingDay: The trading day around which to extract returns.
        marketReturns: A DataFrame of market returns indexed by trading day.
        before: Number of days before the trading day to include.
        after: Number of days after the trading day to include.

    Returns: 
        A DataFrame of returns for the specified window (or None if out of range)
    """

    index = marketReturns.index.searchsorted(tradingDay)
    startIndex = index - before
    endIndex = index + after + 1

    if startIndex < 0 or endIndex > len(marketReturns):
        return None
    
    eventWindow = marketReturns.iloc[startIndex:endIndex].copy()
    eventWindow['offset'] = range(-before, after + 1)

    return eventWindow 

testDay = pd.Timestamp('2022-02-24')
window = getEventWindow(testDay, marketReturns)
print(f'Event window for {testDay.date()}:')
print(window)

Event window for 2022-02-24:
            Return_EEM  Return_GLD  Return_ITA  Return_USO  Return_VIX  offset
Date                                                                          
2022-02-23   -0.011659    0.004507   -0.011375    0.010680    0.076710      -1
2022-02-24   -0.020645   -0.006450    0.027534    0.001510   -0.022566       0
2022-02-25    0.018499   -0.003331    0.030051   -0.008140   -0.090040       1
2022-02-28   -0.013094    0.010365    0.039673    0.025532    0.092787       2
2022-03-01   -0.013268    0.018163   -0.010634    0.064315    0.105141       3


In [11]:
aligned = []

for tradingDay, countRow in dailyCounts.iterrows(): 
    window = getEventWindow(tradingDay, marketReturns)

    if window is None: 
        continue

    for date, returns in window.iterrows(): 
        row = {
            'tradingDay': tradingDay,
            'dayOffset': returns['offset'],
            'events_CaucasusCA' :  countRow['Caucasus and Central Asia'],
            'events_Europe' :  countRow['Europe'],
            'events_MiddleEast' : countRow['Middle East'],
            'events_NorthAfrica': countRow['Northern Africa'],
            'Return_EEM': returns['Return_EEM'],
            'Return_GLD': returns['Return_GLD'],
            'Return_ITA': returns['Return_ITA'],
            'Return_USO': returns['Return_USO'],
            'Return_VIX': returns['Return_VIX'],
        }
        aligned.append(row)

alignedData = pd.DataFrame(aligned)
print(f'Shape: {alignedData.shape}')
alignedData[alignedData['tradingDay'] == '2022-02-24']

Shape: (9390, 11)


,tradingDay,dayOffset,events_CaucasusCA,events_Europe,events_MiddleEast,events_NorthAfrica,Return_EEM,Return_GLD,Return_ITA,Return_USO,Return_VIX
5215,2022-02-24,-1.0,28,400,98,31,-0.011659,0.004507,-0.011375,0.010680,0.076710
5216,2022-02-24,0.0,28,400,98,31,-0.020645,-0.006450,0.027534,0.001510,-0.022566
5217,2022-02-24,1.0,28,400,98,31,0.018499,-0.003331,0.030051,-0.008140,-0.090040
5218,2022-02-24,2.0,28,400,98,31,-0.013094,0.010365,0.039673,0.025532,0.092787
5219,2022-02-24,3.0,28,400,98,31,-0.013268,0.018163,-0.010634,0.064315,0.105141


In [12]:
alignedData.to_csv('../Data/Processed/alignedData.csv', index=False)
print('alignedData saved successfully')

alignedData saved successfully


In [13]:
from Pipeline.align_events import matchToTradingDay, getEventWindow
print('align_events imported successfully')

align_events imported successfully
